# QC-C2-03: OpenQASM 3.0 Transpiler across Qiskit, Cirq, CUDA-Q & QSDK
PESU – C-DAC Quantum Hackathon 2026

**Objective:** Transpile arbitrary OpenQASM 3.0 and OpenQASM 2.0 circuits into standalone, verified Python source code for:
1. **IBM Qiskit** (>= 1.0.0)
2. **Google Cirq** (>= 1.4.0)
3. **NVIDIA CUDA-Q** (>= 0.8.0)
4. **C-DAC Qniverse QSDK** (1.6+)

Includes cross-framework mathematical equivalence verification with automated endianness normalization.

In [ ]:
import os, sys
for candidate in ("src", "../src", "CDAC-Hack/src"):
    if os.path.isdir(os.path.join(candidate, "qnvkit")):
        sys.path.insert(0, os.path.abspath(candidate))
        break

from qnvkit.transpile.api import transpile
from qnvkit.qasm.parser3 import parse_qasm3
from qnvkit.metrics import profile
from qnvkit.decompose.decomposer import decompose

print("qnvkit transpiler ready!")

## 1. Input OpenQASM 3.0 Program
A 3-qubit circuit with superposition, entanglement, parameterized rotations ($RZ(\pi/4), RX(\pi/2)$), and measurement.

In [ ]:
QASM3_SOURCE = """
OPENQASM 3.0;
include "stdgates.inc";

qubit[3] q;
bit[3] c;

// Entanglement
h q[0];
cx q[0], q[1];

// Parameterized gates
rz(pi / 4) q[1];
rx(pi / 2) q[2];
cx q[1], q[2];

// Measurement
c = measure q;
"""

print("OpenQASM 3.0 Source Program:")
print(QASM3_SOURCE.strip())

## 2. Transpile to All 4 Supported Frameworks with Verification
Transpiles to Qiskit, Cirq, CUDA-Q, and QSDK while running local reference simulation and proving mathematical equivalence.

In [ ]:
results = transpile(
    QASM3_SOURCE,
    targets=["qiskit", "cirq", "cudaq", "qniverse"],
    options={"verify": True}
)

for target, res in results.items():
    print(f"=== Target: {target.upper()} ===")
    print(f"Status: {res.report.status}")
    print(f"Target Version: {res.report.target_version}")
    print(f"Equivalence Verified: {res.report.equivalence_verified}")
    print(f"Fidelity: {res.report.fidelity}")
    print("Preview (first 10 lines):")
    for line in res.code.strip().splitlines()[:10]:
        print("  " + line)
    print()

## 3. Seamless Interoperability: C2-03 (Parser) + C1-01 (Decomposer) + C2-03 (CUDA-Q / QSDK Generator)
Demonstrates the unified architecture of `qnvkit`: parse an OpenQASM 3.0 program, decompose it into C-DAC native basis gates `{rx, ry, rz, cz}`, and generate production-grade NVIDIA CUDA-Q code ready for GPU simulation.

In [ ]:
# 1. Parse OpenQASM 3.0 into agnostic CircuitIR
ir = parse_qasm3(QASM3_SOURCE)
print(f"Original Circuit: {len(ir.ops)} operations, depth {profile(ir).depth}")

# 2. Decompose into native basis {rx, ry, rz, cz} via C1-01
decomp_res = decompose(ir, basis={"rx", "ry", "rz", "cz"})
print(f"Decomposition Verified: {decomp_res.verified}")
print(f"Decomposed Circuit: {decomp_res.after.total_gates} gates, depth {decomp_res.after.depth}")

# 3. Transpile decomposed circuit to NVIDIA CUDA-Q via C2-03
from qnvkit.transpile.cudaq_gen import generate_cudaq_code
cudaq_res = generate_cudaq_code(decomp_res.circuit)
print("\n--- Generated Decomposed CUDA-Q Kernel ---")
print(cudaq_res.code)